<font size = "4">

For some of these questions, you are either asked to explicitly add methods to the `Graph` class, or adding the methods to the Graph class would be advantageous to solve the problem.

Here is an example of how you can add a method to the `Graph` class:

In [1]:
from adjacency_graph import Graph

# write function
def check_for_key(self, key):
    if key in self._vertices:
        print("Yep, it's in here!")
    else:
        print(f"Nope, no {key} here!")

# Assign function to a new method of the class
Graph.check_for_key = check_for_key

G = Graph()
G.add_edge("A", "B")

G.check_for_key("A")
G.check_for_key("X")

Yep, it's in here!
Nope, no X here!


<font size = "3">

**(Q1)** Write a function or method that performs a depth-first search (DFS) and a topological sort at the same time. That is, instead of doing DFS and then sorting the vertices in a post-processing step, build up a data structure containing the keys/vertices that produces the topologically sorted order.

In [2]:
def dfs_topo_sort(self) -> list:
    topo_order = []  # 
    def _dfs_visit(vertex):
        vertex.color = "gray"
        for next_vertex in vertex.get_neighbors():
            if next_vertex.color == "white":
                next_vertex.previous = vertex
                _dfs_visit(next_vertex)
            elif next_vertex.color == "gray":
                # raise error if there is a cycle
                raise ValueError("Graph has a cycle")
        # insert
        vertex.color = "black"
        topo_order.insert(0, vertex.key)

    for vertex in self:
        if vertex.color == "white":
            _dfs_visit(vertex)

    return topo_order

<font size = "3">

**(Q2)** Write the `transpose` method for the `Graph` class. If `G` is a directed graph, then `G.transpose()` will return its transpose.

In [6]:
def transpose(self):
    new_graph = Graph()
    # copy all vertex
    for key in self.get_vertices():
        new_graph.set_vertex(key)
    # transpose 
    for from_vertex, to_vertex in self.get_edges():
        weight = self._edges[(from_vertex, to_vertex)] 
        new_graph.add_edge(to_vertex, from_vertex, weight)

    return new_graph

<font size = "3">

**(Q3)** Create a modification of the depth-first search (DFS) method that returns a new graph representing the strongly connected components of the graph. (See the first two images of "lecture_24_b_scc.ipynb" for an example).

In [ ]:
def get_sccs(self) -> list:    
    # first DFS on original graph to get finishing order
    finish_order = []

    # Reset colors
    for vertex in self:
        vertex.color = "white"

    def dfs_finish(vertex):
        vertex.color = "gray"
        for next_vertex in vertex.get_neighbors():
            if next_vertex.color == "white":
                dfs_finish(next_vertex)
        vertex.color = "black"
        finish_order.append(vertex.key)

    for vertex in self:
        if vertex.color == "white":
            dfs_finish(vertex)

    # second DFS on transpose graph in reverse finishing order
    gt = self.transpose()
    for vertex in gt:
        vertex.color = "white"
    sccs = []

    def dfs_collect(vertex, component):
        vertex.color = "gray"
        component.append(vertex.key)
        for next_vertex in vertex.get_neighbors():
            if next_vertex.color == "white":
                dfs_collect(next_vertex, component)
        vertex.color = "black"

    for key in reversed(finish_order):
        vertex = gt.get_vertex(key)
        if vertex.color == "white":
            component = []
            dfs_collect(vertex, component)
            sccs.append(component)

    return sccs

# Build condensed graph
def scc_graph(self) -> "Graph":
    
    sccs = self.get_sccs()

    # map each original vertex key to its SCC label
    vertex_to_component = {}
    component_labels = []

    for component in sccs:
        label = "".join(sorted(component))
        component_labels.append(label)
        for key in component:
            vertex_to_component[key] = label

    # build condensed SCC graph
    new_graph = Graph()

    for label in component_labels:
        new_graph.set_vertex(label)

    for from_key, to_key in self.get_edges():
        from_comp = vertex_to_component[from_key]
        to_comp = vertex_to_component[to_key]

        if from_comp != to_comp:
            if (from_comp, to_comp) not in new_graph._edges:
                new_graph.add_edge(from_comp, to_comp, 0)

    return new_graph

<font size = "3">

**(Q4)** Write a program to solve the following problem: you have two jugs, a 4-gallon and a 3-gallon. Neither of the jugs has any markings. There is a pump that can be used to fill the jugs with water. How can you get exactly two gallons of water in the 4-gallon jug?


**Hint:** Represent this as a graph, where the key of each vertex is an ordered pair of integers $(a, b)$ satisfying

$$0\leq a\leq 4,\quad 0\leq b\leq 3$$



In [12]:
from adjacency_graph import Graph


def build_water_jug_graph() -> Graph:
    """Build a graph of all valid states and transitions for the water jug problem."""
    g = Graph()
    cap_a, cap_b = 4, 3

    for a in range(cap_a + 1):
        for b in range(cap_b + 1):
            state = (a, b)
            transitions = []

            transitions.append((cap_a, b))           # Fill A
            transitions.append((a, cap_b))           # Fill B
            transitions.append((0, b))               # Empty A
            transitions.append((a, 0))               # Empty B

            pour = min(a, cap_b - b)                 # Pour A -> B
            transitions.append((a - pour, b + pour))

            pour = min(b, cap_a - a)                 # Pour B -> A
            transitions.append((a + pour, b - pour))

            for next_state in transitions:
                if next_state != state:  # skip self-loops
                    g.add_edge(state, next_state)

    return g


def solve_water_jug():

    g = build_water_jug_graph()
    start = g.get_vertex((0, 0))

    # BFS
    g.bfs(start)

    # Find any state where the 4-gallon jug has exactly 2 gallons
    for a in range(5):
        for b in range(4):
            if a == 2:
                vertex = g.get_vertex((a, b))
                if vertex and vertex.distance < sys.maxsize:
                    # Trace back the path
                    path = []
                    current = vertex
                    while current:
                        path.append(current.key)
                        current = current.previous
                    path.reverse()
                    print(f"Solution found in {len(path) - 1} steps:")
                    for i, state in enumerate(path):
                        print(f"  Step {i}: 4-gal = {state[0]}, 3-gal = {state[1]}")
                    return path

    print("No solution found")
    return None


import sys
solve_water_jug()

Solution found in 6 steps:
  Step 0: 4-gal = 0, 3-gal = 0
  Step 1: 4-gal = 0, 3-gal = 3
  Step 2: 4-gal = 3, 3-gal = 0
  Step 3: 4-gal = 3, 3-gal = 3
  Step 4: 4-gal = 4, 3-gal = 2
  Step 5: 4-gal = 0, 3-gal = 2
  Step 6: 4-gal = 2, 3-gal = 0


[(0, 0), (0, 3), (3, 0), (3, 3), (4, 2), (0, 2), (2, 0)]

<font size = "3">

**(Q5)** Generalize the problem above so that the parameters to your solution include the size of each jug and the final amount of water to be left in the larger jug.

In [13]:
from adjacency_graph import Graph
import sys

def solve_water_jug(cap_a: int, cap_b: int, target: int) -> list:
    # cap_a: capacity of the larger jug
    # cap_b: capacity of the smaller jug
    # target: desired amount of water in the larger jug

    g = Graph()

    for a in range(cap_a + 1):
        for b in range(cap_b + 1):
            state = (a, b)
            transitions = []

            transitions.append((cap_a, b))           # Fill A
            transitions.append((a, cap_b))           # Fill B
            transitions.append((0, b))               # Empty A
            transitions.append((a, 0))               # Empty B

            pour = min(a, cap_b - b)                 # Pour A -> B
            transitions.append((a - pour, b + pour))

            pour = min(b, cap_a - a)                 # Pour B -> A
            transitions.append((a + pour, b - pour))

            for next_state in transitions:
                if next_state != state: # skip self-loops
                    g.add_edge(state, next_state)

    # BFS from (0, 0)
    start = g.get_vertex((0, 0))
    g.bfs(start)

    # Find the shortest path to any state where jug A has exactly target
    best = None
    for b in range(cap_b + 1):
        vertex = g.get_vertex((target, b))
        if vertex and vertex.distance < sys.maxsize:
            if best is None or vertex.distance < best.distance:
                best = vertex

    if best is None:
        print("No solution found")
        return []

    # Trace back path
    path = []
    current = best
    while current:
        path.append(current.key)
        current = current.previous
    path.reverse()

    print(f"Jugs: {cap_a}-gallon and {cap_b}-gallon, target: {target} in larger jug")
    print(f"Solution in {len(path) - 1} steps:")
    for i, state in enumerate(path):
        print(f"  Step {i}: {cap_a}-gal = {state[0]}, {cap_b}-gal = {state[1]}")

    return path



In [14]:
solve_water_jug(4, 3, 2)
print()

Jugs: 4-gallon and 3-gallon, target: 2 in larger jug
Solution in 6 steps:
  Step 0: 4-gal = 0, 3-gal = 0
  Step 1: 4-gal = 0, 3-gal = 3
  Step 2: 4-gal = 3, 3-gal = 0
  Step 3: 4-gal = 3, 3-gal = 3
  Step 4: 4-gal = 4, 3-gal = 2
  Step 5: 4-gal = 0, 3-gal = 2
  Step 6: 4-gal = 2, 3-gal = 0

